# BTP: explainable AI-content detection, Kaggle GPU runner
Settings: **Accelerator = GPU T4**, **Internet = On**. Then *Save Version -> Save & Run All (Commit)* so it runs in the background. Outputs (models, features, metrics, logs) appear under *Output*.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /kaggle/working
!rm -rf BTP-1 && git clone -q --depth 1 https://github.com/lakshayg2005/BTP-1.git
%cd BTP-1
# big, re-downloadable things go to /kaggle/tmp so the saved output stays small
!mkdir -p /kaggle/tmp/data /kaggle/tmp/hf && ln -sfn /kaggle/tmp/data data
import os
os.environ.update(PY='python', HF_HOME='/kaggle/tmp/hf', TMPDIR_BTP='/kaggle/tmp/t',
                  IMG_TRAIN='3000', IMG_TEST='500', TXT_TRAIN='2000', TXT_TEST='400', BTP_TEXT_LM_SIZE='1.5B')

In [ ]:
!bash scripts/run_phase.sh deps 2>&1 | tail -3
!python -c "import torch, transformers; print(torch.__version__, transformers.__version__, torch.cuda.is_available())"

## Phase 1: image

In [ ]:
!bash scripts/run_phase.sh image 2>&1 | grep -v -E 'it/s\]|s/it\]' | tail -60

## Phase 3: text

In [ ]:
!bash scripts/run_phase.sh text 2>&1 | grep -v -E 'it/s\]|s/it\]' | tail -60

## Sanity check: full pipeline + explanations on a few test images

In [ ]:
import glob, subprocess
for p in sorted(glob.glob('data/images/test/ai/*.png'))[:2] + sorted(glob.glob('data/images/test/real/*.png'))[:2]:
    print('=' * 80, '\n', p)
    print(subprocess.run(['python', '-m', 'scripts.analyze', p, '--explainer', 'llm'], capture_output=True, text=True).stdout[-1500:])

In [ ]:
!ls -la models features && cat models/*.metrics.json